In [ ]:
import glob

import pandas as pd
from rdkit import Chem


def inchi_to_smiles(inchi):
    try:
        mol = Chem.MolFromInchi(inchi)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol)
    except Exception as e:
        print(f"Error converting InChI to SMILES: {e!s}")
        return None


def convert_dataframe_inchi_to_smiles(df, inchi_column="InChI", smiles_column="smiles"):
    """
    Convert InChI strings in a DataFrame to SMILES.

    :param df: Input DataFrame
    :param inchi_column: Name of the column containing InChI strings
    :param smiles_column: Name of the new column to store SMILES strings
    :return: DataFrame with added SMILES column
    """
    if inchi_column not in df.columns:
        raise ValueError(f"Column '{inchi_column}' not found in DataFrame")

    # Apply the conversion function to the InChI column and drop inchi column
    df[smiles_column] = df[inchi_column].apply(inchi_to_smiles)
    df.drop(columns=[inchi_column], inplace=True)
    df = df[["smiles", "rt"]]
    return df


datasets = glob.glob("../../data/dataset/10_subdataset/*.xlsx")
name_datasets = [data.split("/")[-1].split(".")[0] for data in datasets]

for name, path in zip(name_datasets, datasets):
    df = pd.read_excel(path, usecols=["InChI", "RT"])
    # remove index
    df = df.reset_index(drop=True).rename(columns={"RT": "rt"})[["InChI", "rt"]]
    df["rt"] = df["rt"] * 60  # convert to seconds

    df_with_smiles = convert_dataframe_inchi_to_smiles(df)
    print("\nDataFrame with SMILES:")
    print(f"{name} dataset")
    # save
    df_with_smiles.to_csv(
        f"../../data/dataset/10_subdataset_smiles/{name}.txt",
        sep="\t",
        index=False,
    )

    df_with_smiles.head()
